### Ingestión de la carpeta "production_country"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

###Paso 1 - Leer los archivos JSON usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

production_countries_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("countryId", IntegerType(), True)
])


In [0]:
#production_countries_df = spark.read.schema(production_countries_schema).option("multiLine",True).json("abfss://bronze@moviee.dfs.core.windows.net/production_country")

production_countries_df = spark.read.schema(production_countries_schema).option("multiLine",True).json(f"{bronze_folder_path}/production_country")





In [0]:
production_countries_df.printSchema()

In [0]:
display(production_countries_df)

In [0]:
production_countries_df.count()

### Paso 2 - Renombrar las columnas y añadir columnas nuevas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

#production_countries_final_df = production_countries_df \
#                        .withColumnsRenamed({"movieId": "movie_id",
#                                             "countryId": "country_id"}) \
#                        .withColumn("ingestion_date", current_timestamp()) \
#                        .withColumn("environment", lit("production"))


production_countries_final_df = production_countries_df \
                                  .withColumnsRenamed({"movieId": "movie_id",
                                             "countryId": "country_id"})

production_countries_final_df = add_ingestion_date(production_countries_final_df) \
                                  .withColumn("environment", lit(v_environment))


In [0]:
display(production_countries_final_df)

### Paso 3 - Escribir la salida en un formato "Parquet"

In [0]:
production_countries_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/production_countries")

In [0]:
display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/production_countries"))

### Paso 4 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 3, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 3, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 3

In [0]:
production_countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.productions_countries")

In [0]:
%sql
SELECT * FROM movie_silver.productions_countries

In [0]:
dbutils.notebook.exit("Success")